1 — Import Libraries and Define Paths

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_CSV_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "swift_payments_raw_export.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 25_000

print("Project root:", PROJECT_ROOT)
print("Raw CSV found:", RAW_CSV_PATH.exists())
print("Chunk size:", CHUNK_SIZE)

2 — Validate Dataset Schema

In [ ]:
EXPECTED_COLUMNS = [
    "DISCLAIMER",
    "EPOCH_SECONDS",
    "CURRENCY_CODE",
    "USD_AMOUNT",
    "ORDERING_BIC8",
    "BENEFICIARY_BIC8",
    "BIC8_PATH",
    "TOTAL_DURATION",
    "CONTINUITY_QFLAG",
    "CURRENCY_CONVERSION_COUNT",
    "PARTICIPANT_COUNT",
    "TRANSACTION_TYPE_CODE",
    "ORDERING_ACCOUNT",
    "BENEFICIARY_ACCOUNT",
    "ORDERING_DETAILS",
    "ORDERING_TOKENS",
    "BENEFICIARY_DETAILS",
    "BENEFICIARY_TOKENS",
    "ORDERING_TYPE",
    "BENEFICIARY_TYPE",
    "ANOMALY_TYPE"
]

actual_columns = pd.read_csv(
    RAW_CSV_PATH,
    encoding="utf-8-sig",
    nrows=0
).columns.tolist()

print("Expected columns:", len(EXPECTED_COLUMNS))
print("Actual columns:", len(actual_columns))

missing_columns = set(EXPECTED_COLUMNS) - set(actual_columns)
extra_columns = set(actual_columns) - set(EXPECTED_COLUMNS)

print("Missing columns:", missing_columns)
print("Extra columns:", extra_columns)

assert actual_columns == EXPECTED_COLUMNS, (
    "Dataset schema does not match the expected columns."
)

print("Schema validation passed.")

3 — Define Audit Rules

In [ ]:
NUMERIC_COLUMNS = [
    "EPOCH_SECONDS",
    "USD_AMOUNT",
    "TOTAL_DURATION",
    "CURRENCY_CONVERSION_COUNT",
    "PARTICIPANT_COUNT"
]

CATEGORICAL_COLUMNS = [
    "CURRENCY_CODE",
    "CONTINUITY_QFLAG",
    "TRANSACTION_TYPE_CODE",
    "ORDERING_TYPE",
    "BENEFICIARY_TYPE",
    "ANOMALY_TYPE"
]

EXPECTED_CATEGORIES = {
    "CONTINUITY_QFLAG": {"Y", "N"},
    "TRANSACTION_TYPE_CODE": {
        "DIRECT", "INDIRECT", "COVERED"
    },
    "ORDERING_TYPE": {"B", "C"},
    "BENEFICIARY_TYPE": {"B", "C"},
    "ANOMALY_TYPE": {
        "ROMANCE_SCAM",
        "CEO_SCAM",
        "ACCOUNT_REPLACEMENT_SCAM",
        "FAT_FINGER_MISTAKE",
        "DUPLICATE_PAYMENT"
    }
}

print("Numeric columns:", len(NUMERIC_COLUMNS))
print("Categorical columns:", len(CATEGORICAL_COLUMNS))
print("Audit rules initialized.")

4 — Run the Full Data Quality Audit

In [ ]:
from pandas.util import hash_pandas_object

total_rows = 0
blank_counts = Counter()
whitespace_counts = Counter()
category_counts = defaultdict(Counter)

numeric_invalid = Counter()
numeric_negative = Counter()
numeric_zero = Counter()
numeric_min = {}
numeric_max = {}

path_gap_count = 0
continuity_mismatch_count = 0
candidate_duplicate_count = 0
seen_row_hashes = set()

reader = pd.read_csv(
    RAW_CSV_PATH,
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
    na_filter=False,
    chunksize=CHUNK_SIZE
)

for chunk_number, chunk in enumerate(reader, start=1):

    total_rows += len(chunk)

    # Check blank values and whitespace inconsistencies
    for col in EXPECTED_COLUMNS:
        values = chunk[col].astype(str)
        stripped = values.str.strip()

        blank_counts[col] += int(stripped.eq("").sum())

        whitespace_counts[col] += int(
            values.ne(stripped).sum()
        )

    # Inspect categorical distributions
    for col in CATEGORICAL_COLUMNS:
        counts = chunk[col].value_counts(dropna=False)
        category_counts[col].update(counts.to_dict())

    # Inspect numeric validity and ranges
    for col in NUMERIC_COLUMNS:
        raw = chunk[col].str.strip()
        values = pd.to_numeric(raw, errors="coerce")

        nonblank = raw.ne("")
        invalid = nonblank & values.isna()

        numeric_invalid[col] += int(invalid.sum())
        numeric_negative[col] += int((values < 0).sum())
        numeric_zero[col] += int((values == 0).sum())

        if values.notna().any():
            current_min = float(values.min())
            current_max = float(values.max())

            numeric_min[col] = min(
                numeric_min.get(col, current_min),
                current_min
            )

            numeric_max[col] = max(
                numeric_max.get(col, current_max),
                current_max
            )

    # Inspect routing path gaps
    has_gap = chunk["BIC8_PATH"].str.contains(
        "...", regex=False, na=False
    )

    continuity = (
        chunk["CONTINUITY_QFLAG"]
        .str.strip()
        .str.upper()
    )

    path_gap_count += int(has_gap.sum())

    continuity_mismatch_count += int(
        ((has_gap) & (continuity == "Y")).sum()
        +
        ((~has_gap) & (continuity == "N")).sum()
    )

    # Identify possible exact duplicate row signatures
    row_hashes = hash_pandas_object(
        chunk[EXPECTED_COLUMNS],
        index=False
    )

    for row_hash in row_hashes:
        signature = int(row_hash)

        if signature in seen_row_hashes:
            candidate_duplicate_count += 1
        else:
            seen_row_hashes.add(signature)

    print(
        f"Chunk {chunk_number}: "
        f"{total_rows:,} rows audited"
    )

print("\nData audit completed.")
print("Total records:", f"{total_rows:,}")
print("Candidate duplicate signatures:",
      f"{candidate_duplicate_count:,}")
print("Routing paths with gaps:", f"{path_gap_count:,}")
print("Continuity mismatches:",
      f"{continuity_mismatch_count:,}")

5 — Generate Audit Reports

In [ ]:
column_profile = pd.DataFrame([
    {
        "COLUMN": col,
        "BLANK_COUNT": blank_counts[col],
        "BLANK_PERCENT": round(
            blank_counts[col] / total_rows * 100, 4
        ) if total_rows else 0,
        "WHITESPACE_ISSUES": whitespace_counts[col],
        "NUMERIC_INVALID": numeric_invalid[col],
        "NEGATIVE_VALUES": numeric_negative[col],
        "ZERO_VALUES": numeric_zero[col],
        "MIN_VALUE": numeric_min.get(col, np.nan),
        "MAX_VALUE": numeric_max.get(col, np.nan)
    }
    for col in EXPECTED_COLUMNS
])

category_rows = []

for col, counts in category_counts.items():
    for value, count in counts.items():
        category_rows.append({
            "COLUMN": col,
            "RAW_VALUE": value,
            "COUNT": count,
            "PERCENT": round(
                count / total_rows * 100, 4
            ) if total_rows else 0
        })

category_profile = pd.DataFrame(category_rows)

audit_summary = pd.DataFrame([
    {"METRIC": "Total records", "VALUE": total_rows},
    {
        "METRIC": "Candidate duplicate signatures",
        "VALUE": candidate_duplicate_count
    },
    {
        "METRIC": "Routing paths with gaps",
        "VALUE": path_gap_count
    },
    {
        "METRIC": "Continuity mismatches",
        "VALUE": continuity_mismatch_count
    }
])

column_profile.to_csv(
    OUTPUT_DIR / "audit_column_profile.csv",
    index=False
)

category_profile.to_csv(
    OUTPUT_DIR / "audit_category_profile.csv",
    index=False
)

audit_summary.to_csv(
    OUTPUT_DIR / "audit_summary.csv",
    index=False
)

print("Audit reports exported successfully.")

display(audit_summary)
display(column_profile)

6 — Identify Unexpected Categories

In [ ]:
unexpected_categories = []

for col, allowed_values in EXPECTED_CATEGORIES.items():

    for raw_value, count in category_counts[col].items():

        standardized = str(raw_value).strip().upper()

        if standardized == "":
            continue

        if standardized not in allowed_values:
            unexpected_categories.append({
                "COLUMN": col,
                "RAW_VALUE": raw_value,
                "STANDARDIZED_VALUE": standardized,
                "COUNT": count
            })

unexpected_df = pd.DataFrame(
    unexpected_categories,
    columns=[
        "COLUMN",
        "RAW_VALUE",
        "STANDARDIZED_VALUE",
        "COUNT"
    ]
)

unexpected_df.to_csv(
    OUTPUT_DIR / "audit_unexpected_categories.csv",
    index=False
)

display(unexpected_df)

if unexpected_df.empty:
    print("No unexpected categorical values found.")
else:
    print("Unexpected categorical values require review.")

7 — Create the Data Issues Log

In [ ]:
# ============================================================
# Cell 7 - Create the Data Issues Log
# Purpose:
# Document all identified data quality issues, cleaning
# decisions, and reasons without modifying the original data.
# ============================================================

issues = []

# ------------------------------------------------------------
# 1. Check missing values, whitespace, and numeric issues
# ------------------------------------------------------------

for col in EXPECTED_COLUMNS:

    # Check blank values
    if blank_counts[col] > 0:
        issues.append({
            "COLUMN": col,
            "ISSUE": "Blank values detected",
            "COUNT": blank_counts[col],
            "DECISION": "Preserve and investigate",
            "REASON": (
                "Missing values may have business meaning. "
                "Do not automatically fill or remove them."
            )
        })

    # Check leading or trailing whitespace
    if whitespace_counts[col] > 0:
        issues.append({
            "COLUMN": col,
            "ISSUE": "Leading or trailing whitespace",
            "COUNT": whitespace_counts[col],
            "DECISION": "Standardize analysis copy only",
            "REASON": (
                "Whitespace normalization may improve consistency. "
                "Original field values must remain unchanged."
            )
        })

    # Check invalid numeric representations
    if numeric_invalid[col] > 0:
        issues.append({
            "COLUMN": col,
            "ISSUE": "Invalid numeric representation",
            "COUNT": numeric_invalid[col],
            "DECISION": "Flag without imputation",
            "REASON": (
                "Invalid numeric values require investigation. "
                "The correct values must not be assumed."
            )
        })


# ------------------------------------------------------------
# 2. Record routing path gaps
# ------------------------------------------------------------

if path_gap_count > 0:
    issues.append({
        "COLUMN": "BIC8_PATH",
        "ISSUE": "Routing path gaps detected",
        "COUNT": path_gap_count,
        "DECISION": "Preserve",
        "REASON": (
            "Path placeholders such as '...' may represent "
            "hidden or discontinuous routing paths. "
            "These records are relevant to routing analysis."
        )
    })


# ------------------------------------------------------------
# 3. Record path continuity inconsistencies
# ------------------------------------------------------------

if continuity_mismatch_count > 0:
    issues.append({
        "COLUMN": "CONTINUITY_QFLAG",
        "ISSUE": "Potential path continuity inconsistencies",
        "COUNT": continuity_mismatch_count,
        "DECISION": "Preserve and investigate",
        "REASON": (
            "The continuity flag may not match the observed "
            "routing path gap indicator. Further investigation "
            "is required before classifying these as errors."
        )
    })


# ------------------------------------------------------------
# 4. Record possible duplicate records
# ------------------------------------------------------------

if candidate_duplicate_count > 0:
    issues.append({
        "COLUMN": "ALL_COLUMNS",
        "ISSUE": "Candidate duplicate row signatures",
        "COUNT": candidate_duplicate_count,
        "DECISION": "Preserve and investigate",
        "REASON": (
            "Repeated row hashes may indicate duplicate records. "
            "Hash matches require verification before concluding "
            "that records are identical. Duplicate payments may "
            "also be part of the dataset's anomaly-generating logic."
        )
    })


# ------------------------------------------------------------
# 5. Record undocumented categorical values
# ------------------------------------------------------------

# This section uses unexpected_df from Cell 6.
# It records categories found in the dataset but not listed
# in the predefined official category reference.

if not unexpected_df.empty:

    for _, row in unexpected_df.iterrows():

        issues.append({
            "COLUMN": row["COLUMN"],
            "ISSUE": (
                f"Undocumented categorical value: "
                f"{row['RAW_VALUE']}"
            ),
            "COUNT": int(row["COUNT"]),
            "DECISION": "Preserve and investigate",
            "REASON": (
                "This category exists in the dataset but is not "
                "listed in the official feature appendix. "
                "It must not be deleted, renamed, or merged "
                "with another category without evidence."
            )
        })


# ------------------------------------------------------------
# 6. Generate the Data Issues Log
# ------------------------------------------------------------

issues_log = pd.DataFrame(
    issues,
    columns=[
        "COLUMN",
        "ISSUE",
        "COUNT",
        "DECISION",
        "REASON"
    ]
)


# ------------------------------------------------------------
# 7. Export the Data Issues Log
# ------------------------------------------------------------

output_path = OUTPUT_DIR / "audit_issues_log.csv"

issues_log.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 8. Display the Audit Results
# ------------------------------------------------------------

print("=" * 60)
print("DATA QUALITY ISSUES LOG")
print("=" * 60)

print(f"Total issue categories recorded: {len(issues_log)}")
print(f"Report saved to: {output_path}")

print("\nData Quality Issues:")
display(issues_log)


# ------------------------------------------------------------
# 9. Verify undocumented anomaly categories
# ------------------------------------------------------------

customer_recall_issues = issues_log[
    issues_log["ISSUE"].str.contains(
        "CUSTOMER_RECALL",
        case=False,
        na=False
    )
]

if not customer_recall_issues.empty:

    print("\nUndocumented anomaly detected:")
    print("Category: CUSTOMER_RECALL")

    print(
        "Records:",
        int(customer_recall_issues["COUNT"].sum())
    )

    print(
        "Decision: Preserve and investigate "
        "during anomaly analysis."
    )

else:
    print(
        "\nNo CUSTOMER_RECALL category was recorded "
        "in the issues log."
    )


# ------------------------------------------------------------
# 10. Completion message
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("Data Issues Log completed successfully.")
print("Original dataset remains unchanged.")
print("=" * 60)

8 — Verify Original Excel Structure

In [ ]:
from openpyxl import load_workbook

EXCEL_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "swift_umdac_datathon_2026_fictitious_payment_data.xlsx"
)

workbook = load_workbook(
    EXCEL_PATH,
    read_only=True,
    data_only=True
)

print("Original Excel Worksheets:")

for sheet in workbook.worksheets:
    print("Sheet:", sheet.title)
    print("Reported Rows:", sheet.max_row)
    print("Reported Columns:", sheet.max_column)

workbook.close()

print("\nCSV Data Rows:", total_rows)

9 — Compare Original Excel and CSV Samples

In [ ]:
# ============================================================
# Cell 9 - Verify Excel to CSV Sample Consistency
# Purpose:
# Compare selected records between the original Excel file
# and exported CSV without modifying either source.
# ============================================================

from openpyxl import load_workbook
import pandas as pd

wb = load_workbook(
    EXCEL_PATH,
    read_only=True,
    data_only=True
)

ws = wb.worksheets[0]

# Compare the first 100 data records
excel_rows = list(
    ws.iter_rows(
        min_row=2,
        max_row=101,
        values_only=True
    )
)

excel_sample = pd.DataFrame(
    excel_rows,
    columns=EXPECTED_COLUMNS
)

csv_sample = pd.read_csv(
    RAW_CSV_PATH,
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
    na_filter=False,
    nrows=100
)

# Compare selected business-sensitive columns
columns_to_check = [
    "EPOCH_SECONDS",
    "CURRENCY_CODE",
    "USD_AMOUNT",
    "ORDERING_ACCOUNT",
    "BENEFICIARY_ACCOUNT",
    "BIC8_PATH",
    "TOTAL_DURATION",
    "ORDERING_DETAILS",
    "ORDERING_TOKENS",
    "BENEFICIARY_DETAILS",
    "BENEFICIARY_TOKENS",
    "ANOMALY_TYPE"
]

comparison_results = []

for col in columns_to_check:

    excel_values = (
        excel_sample[col]
        .fillna("")
        .astype(str)
    )

    csv_values = csv_sample[col].astype(str)

    matches = excel_values.eq(csv_values)

    comparison_results.append({
        "COLUMN": col,
        "MATCHED_ROWS": int(matches.sum()),
        "MISMATCHED_ROWS": int((~matches).sum())
    })

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df)

wb.close()

print("Excel vs CSV sample verification completed.")

In [ ]:
# ============================================================
# Cell 10 - Investigate USD Amount Mismatches
# Purpose:
# Identify discrepancies between Excel and CSV amount
# representations without modifying the original data.
# ============================================================

import pandas as pd
from decimal import Decimal, InvalidOperation

excel_amounts = (
    excel_sample["USD_AMOUNT"]
    .fillna("")
    .astype(str)
)

csv_amounts = (
    csv_sample["USD_AMOUNT"]
    .fillna("")
    .astype(str)
)

# Identify rows with different string representations
mismatch_mask = excel_amounts.ne(csv_amounts)

mismatch_details = pd.DataFrame({
    "EXCEL_ROW_NUMBER": excel_sample.index + 2,
    "EXCEL_AMOUNT": excel_amounts,
    "CSV_AMOUNT": csv_amounts
})

mismatch_details = mismatch_details.loc[
    mismatch_mask
].copy()


# Compare numeric values using Decimal
def parse_decimal(value):
    try:
        return Decimal(str(value).strip())
    except (InvalidOperation, ValueError):
        return None


mismatch_details["NUMERICALLY_EQUAL"] = [
    parse_decimal(excel_value) == parse_decimal(csv_value)
    for excel_value, csv_value in zip(
        mismatch_details["EXCEL_AMOUNT"],
        mismatch_details["CSV_AMOUNT"]
    )
]


# Display mismatch investigation results
print("=" * 60)
print("USD AMOUNT MISMATCH INVESTIGATION")
print("=" * 60)

print("String mismatches:", len(mismatch_details))

display(mismatch_details)

print(
    "Numerically equivalent mismatches:",
    int(mismatch_details["NUMERICALLY_EQUAL"].sum())
)

print(
    "Potential numeric discrepancies:",
    int((~mismatch_details["NUMERICALLY_EQUAL"]).sum())
)

In [ ]:
# Cell 11 - Document Excel to CSV Sample Verification

verification_record = pd.DataFrame([
    {
        "CHECK": "Excel to CSV sample comparison",
        "SAMPLE_SIZE": 100,
        "STRING_MISMATCHES": 3,
        "NUMERICALLY_EQUIVALENT": 3,
        "POTENTIAL_NUMERIC_DISCREPANCIES": 0,
        "RESULT": "PASSED",
        "LIMITATION": "Sample validation only"
    }
])

verification_record.to_csv(
    OUTPUT_DIR / "audit_excel_csv_verification.csv",
    index=False
)

display(verification_record)

print("Excel to CSV sample verification documented.")